# Portfolio Metrics Extraction PoC

Notebook for extracting metrics from portfolio-company PDF reports. The default path is direct text parsing, then OCR, then GLM-OCR if the page is still unclear. Results are normalized into a long table, written to SQLite, and reviewed with simple historical checks.

## 1. Assumptions and Scope

- One PDF usually maps to one company and one reporting period.
- The PoC tracks a narrow metric set: revenue, ARR, EBITDA, gross margin, churn, headcount, cash balance, pipeline, and a short commentary summary.
- Direct PDF text extraction is the first pass because it is faster and easier to audit.
- OCR is the fallback for scanned PDFs or damaged text layers.
- GLM-OCR is the last pass when the earlier steps still leave missing or suspicious metrics.

## 2. Architecture Diagram

```mermaid
flowchart TD
    A[Folder of PDF reports] --> B[Batch iterator]
    B --> C[LangGraph StateGraph]
    C --> D[Layer 1: direct PDF text extraction<br/>pdfplumber + pypdf]
    D --> E{Enough metrics found?}
    E -->|yes| G[Validation + normalization]
    E -->|no| F[Layer 2: OCR fallback<br/>pdf2image + pytesseract]
    F --> H{Enough metrics found?}
    H -->|yes| G
    H -->|no| I[Layer 3: Ollama GLM-OCR<br/>local VLM fallback]
    I --> G
    G --> J[Pandas long table]
    J --> K[SQLite persistence]
    K --> L[Historical checks]
    L --> M[Simple forecast baseline<br/>SQLite + Ridge regression]
    J --> N[Review table + basic charts]
```

## 3. Dependency Setup

The notebook installs the Python packages it needs. `scikit-learn` is used for the simple forecast baseline. Tesseract for standard OCR. Ollama is a separate CLI and must be available on PATH for the GLM-OCR (VLM-based) cells.

macOS:

```bash
brew install tesseract poppler
```

Ubuntu / Debian:

```bash
sudo apt-get update
sudo apt-get install -y tesseract-ocr poppler-utils
```

Ollama:

- macOS: install from the official download page, or use the desktop app.
- Linux:

```bash
curl -fsSL https://ollama.com/install.sh | sh
ollama serve
```

- Pull and test the model:

```bash
ollama pull glm-ocr
ollama run glm-ocr Text Recognition: ./image.png
```

The notebook was run with Python 3.12.5.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pdfplumber": "pdfplumber",
    "pypdf": "pypdf",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "pydantic": "pydantic",
    "langchain": "langchain",
    "langchain_core": "langchain-core",
    "langgraph": "langgraph",
    "langchain_openai": "langchain-openai",
    "pdf2image": "pdf2image",
    "pytesseract": "pytesseract",
    "sklearn": "scikit-learn",
}

missing = [pkg for module, pkg in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required Python packages are available.")

ollama_path = shutil.which("ollama")
if ollama_path is None:
    print("Ollama CLI not found on PATH. Install Ollama before running the GLM-OCR cells.")
else:
    print(f"Ollama CLI found at: {ollama_path}")

## 4. Configuration

The notebook runs directly against the PDFs in the local challenge package.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sqlite3
import sys
import warnings
from pathlib import Path
from typing import Any, Literal, Required, TypedDict

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
SQLITE_PATH = OUTPUT_DIR / "portfolio_metrics.sqlite"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIGURED_PDF_DIRS = [
    Path(os.environ[var]).expanduser()
    for var in ("TECH_CHALLENGE_PACKAGE_DIR", "PDF_INPUT_DIR")
    if os.getenv(var)
]
PDF_CANDIDATES = [*CONFIGURED_PDF_DIRS, PROJECT_ROOT / "Tech Challenge Package"]
PDF_DIR = next((p for p in PDF_CANDIDATES if p.exists() and list(p.glob("*.pdf"))), None)
if PDF_DIR is None:
    raise FileNotFoundError("Could not find the challenge PDF folder. Set TECH_CHALLENGE_PACKAGE_DIR or place 'Tech Challenge Package' in the repo root.")
PDF_INPUT_DIR = PDF_DIR

print(f"Python runtime: {sys.version.split()[0]}")
print("Challenge PDF package detected locally.")
print("Main input directory set from local configuration.")

## 5. Input PDFs

The notebook runs directly against the challenge package PDFs.

In [ ]:
pdf_paths = sorted(PDF_INPUT_DIR.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError("No PDFs found in the configured challenge PDF folder.")

print(f"Found {len(pdf_paths)} PDFs:")
display(pd.DataFrame({"pdf_file": [p.name for p in pdf_paths]}))

## 6. Metric Schemas

The `MetricExtractionResult` schema is also the contract used by the LangChain structured extraction fallback.

In [ ]:
class ExtractedMetric(BaseModel):
    metric_name: str = Field(description="Canonical metric name")
    metric_value: float | None = Field(description="Normalized numeric value")
    unit: str | None = Field(description="Unit such as USD, CAD, %, count")
    evidence_snippet: str | None = Field(description="Short source snippet supporting the extraction")
    confidence: float = Field(description="Confidence from 0 to 1", ge=0, le=1)


class MetricExtractionResult(BaseModel):
    company_name: str | None
    report_date: str | None
    metrics: list[ExtractedMetric]
    commentary_summary: str | None
    needs_review: bool
    review_reason: str | None


class MetricRecord(BaseModel):
    company_name: str | None
    report_date: str | None
    metric_name: str
    metric_value: float | None
    unit: str | None
    extraction_method: str
    confidence: float
    source_file: str
    source_page: int | None
    evidence_snippet: str | None
    flagged_for_review: bool = False
    review_reason: str | None = None


METRIC_ALIASES = {
    "revenue": [
        "total recognized revenue", "recognized revenue", "net revenue", "quarterly revenue",
        "platform revenue", "gross transaction revenue", "total billings", "sales", "revenue",
    ],
    "arr": [
        "contracted annual recurring revenue", "annual recurring revenue", "contracted arr",
        "end-of-period arr", "subscription arr", "arr",
    ],
    "ebitda": ["adjusted ebitda", "ebitda"],
    "gross_margin": ["gross margin %", "gross margin"],
    "churn": ["logo churn rate", "annual logo churn", "logo churn", "churn", "net revenue retention"],
    "headcount": ["total headcount", "headcount", "employees", "fte"],
    "cash_balance": ["cash & restricted cash", "cash & equivalents", "cash balance", "cash on hand", "cash"],
    "pipeline": ["weighted pipeline", "sales pipeline", "total pipeline", "pipeline", "weighted acv", "wtd. acv"],
}

TARGET_METRICS = list(METRIC_ALIASES.keys())
MONEY_METRICS = {"revenue", "arr", "ebitda", "cash_balance", "pipeline"}
PERCENT_METRICS = {"gross_margin", "churn"}
COUNT_METRICS = {"headcount"}

print("Target metrics:", TARGET_METRICS)

## 7. Regex and Normalization Utilities

Key choices:

- store currency-like metrics as normalized absolute values, e.g. `$1.2M` becomes `1200000.0`;
- store percentages as their displayed percentage value, e.g. `5.8%` becomes `5.8` with unit `%`;
- prefer exact/total/current-period labels over incidental mentions;
- preserve source page and evidence text.

In [ ]:
VALUE_PATTERN = re.compile(
    r"""
    (?P<raw>
        (?:C\$|US\$|\$|USD|CAD)?\s*
        \(?\s*[-+]?\s*
        \d[\d,]*(?:\.\d+)?
        \s*(?:%|billion|bn|million|mm|mn|thousand|b|m|k)?
        \s*\)?
    )
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)


def clean_text(text: str) -> str:
    text = text.replace("\u2212", "-").replace("\u2013", "-").replace("\u2014", "-")
    text = text.replace("(cid:127)", "-")
    return re.sub(r"[ \t]+", " ", text).strip()


def compact_whitespace(text: str) -> str:
    return re.sub(r"\s+", " ", clean_text(text)).strip()


def alias_regex(alias: str) -> re.Pattern:
    escaped = re.escape(alias).replace(r"\ ", r"\s+")
    return re.compile(rf"(?<![A-Za-z0-9]){escaped}(?![A-Za-z0-9])", re.IGNORECASE)


def parse_numeric_value(raw: str, metric_name: str) -> tuple[float | None, str | None]:
    raw_clean = clean_text(raw)
    currency = None
    if re.search(r"\bCAD\b|C\$", raw_clean, flags=re.IGNORECASE):
        currency = "CAD"
    elif "$" in raw_clean or re.search(r"\bUSD\b|US\$", raw_clean, flags=re.IGNORECASE):
        currency = "USD"

    number_match = re.search(r"\d[\d,]*(?:\.\d+)?", raw_clean)
    if not number_match:
        return None, None

    value = float(number_match.group(0).replace(",", ""))
    if "(" in raw_clean and ")" in raw_clean or re.search(r"(^|\s)-\s*\d", raw_clean):
        value *= -1

    suffix_match = re.search(
        r"(%|billion|bn|million|mm|mn|thousand|b|m|k)\s*\)?\s*$",
        raw_clean,
        flags=re.IGNORECASE,
    )
    suffix = suffix_match.group(1).lower() if suffix_match else None

    if suffix in {"b", "bn", "billion"}:
        value *= 1_000_000_000
    elif suffix in {"m", "mm", "mn", "million"}:
        value *= 1_000_000
    elif suffix in {"k", "thousand"}:
        value *= 1_000

    if suffix == "%" or "%" in raw_clean:
        unit = "%"
    elif metric_name in COUNT_METRICS:
        unit = "count"
    elif metric_name in MONEY_METRICS:
        unit = currency or "USD"
    else:
        unit = currency

    return value, unit


def reject_candidate(metric_name: str, line: str, raw_value: str, value: float | None, unit: str | None) -> bool:
    line_l = line.lower()
    if value is None:
        return True
    if metric_name in MONEY_METRICS and unit == "%":
        return True
    if metric_name in PERCENT_METRICS and unit != "%":
        return True
    if metric_name == "revenue" and any(term in line_l for term in ["revenue growth", "as % of revenue", "spend as %", "retention"]):
        return True
    if metric_name == "arr" and any(term in line_l for term in ["arr growth", "arr per", "from existing accounts", "% of total arr", "expansion arr as"]):
        return True
    if metric_name == "cash_balance":
        if any(term in line_l for term in ["non-cash", "depreciation", "amortisation", "amortization"]):
            return True
        if "runway" in line_l and not any(term in line_l for term in ["cash balance", "cash &", "cash on hand"]):
            return True
    if metric_name == "pipeline" and unit == "%":
        return True
    return False


def score_candidate(metric_name: str, alias: str, line: str, method: str, unit: str | None) -> float:
    base = {"direct_parse": 0.86, "ocr": 0.68, "glm_ocr": 0.64, "llm_vlm": 0.56}.get(method, 0.60)
    line_l = line.lower()
    score = base
    if len(alias) >= 10:
        score += 0.04
    if metric_name == "revenue" and any(term in line_l for term in ["total", "recognized", "quarterly", "net revenue", "platform revenue"]):
        score += 0.05
    if metric_name == "arr" and any(term in line_l for term in ["contracted", "end-of-period", "subscription", "annual recurring revenue"]):
        score += 0.05
    if metric_name == "headcount" and "total" in line_l:
        score += 0.04
    if metric_name == "cash_balance" and any(term in line_l for term in ["cash balance", "cash &", "cash on hand"]):
        score += 0.05
    if metric_name in MONEY_METRICS and unit in {"USD", "CAD"}:
        score += 0.02
    if "footnote" in line_l or line_l.startswith("("):
        score -= 0.08
    return max(0.0, min(score, 0.98))


def extract_pipeline_from_text(pages: list[dict[str, Any]], method: str) -> dict[str, Any] | None:
    candidates: list[dict[str, Any]] = []
    full_text = compact_whitespace("\n".join(page.get("text", "") for page in pages))

    direct_match = re.search(
        r"(weighted\s+pipeline\s+(?:stands\s+at|was|is|of)?\s*)"
        r"((?:C\$|US\$|\$|USD|CAD)?\s*\d[\d,]*(?:\.\d+)?\s*(?:million|mm|mn|m|billion|bn|b|k|thousand)?)",
        full_text,
        flags=re.IGNORECASE,
    )
    if direct_match:
        raw = direct_match.group(2)
        value, unit = parse_numeric_value(raw, "pipeline")
        if value is not None:
            candidates.append({
                "metric_name": "pipeline",
                "metric_value": value,
                "unit": unit,
                "extraction_method": method,
                "confidence": 0.84 if method == "direct_parse" else 0.64,
                "source_page": None,
                "evidence_snippet": direct_match.group(0)[:300],
            })

    for page in pages:
        lines = [clean_text(line) for line in page.get("text", "").splitlines()]
        for i, line in enumerate(lines):
            line_l = line.lower()
            if "stage" not in line_l or not any(term in line_l for term in ["weighted", "wtd", "total pipeline", "acv"]):
                continue
            rows = []
            row_values = []
            for row in lines[i + 1 : i + 8]:
                row_l = row.lower().strip()
                if not row_l or row_l.startswith("(") or row_l.startswith("-") or row_l in {"key risks", "commentary"}:
                    break
                values = list(VALUE_PATTERN.finditer(row))
                money_values = []
                for match in values:
                    value, unit = parse_numeric_value(match.group("raw"), "pipeline")
                    if unit in {"USD", "CAD"} and value is not None:
                        money_values.append((value, unit, match.group("raw")))
                if money_values:
                    rows.append(row)
                    row_values.append(money_values[-1])
                elif rows:
                    break
            if row_values:
                total_value = sum(v for v, _, _ in row_values)
                unit = row_values[0][1]
                candidates.append({
                    "metric_name": "pipeline",
                    "metric_value": total_value,
                    "unit": unit,
                    "extraction_method": method,
                    "confidence": 0.80 if method == "direct_parse" else 0.60,
                    "source_page": page.get("page_number"),
                    "evidence_snippet": compact_whitespace(" | ".join([line, *rows]))[:400],
                })
    if not candidates:
        return None
    return sorted(candidates, key=lambda c: c["confidence"], reverse=True)[0]


def extract_metrics_from_pages(pages: list[dict[str, Any]], method: str = "direct_parse") -> list[dict[str, Any]]:
    best_by_metric: dict[str, dict[str, Any]] = {}
    for page in pages:
        for raw_line in page.get("text", "").splitlines():
            line = clean_text(raw_line)
            if not line:
                continue
            for metric_name, aliases in METRIC_ALIASES.items():
                if metric_name == "pipeline":
                    continue
                for alias in sorted(aliases, key=len, reverse=True):
                    match = alias_regex(alias).search(line)
                    if not match:
                        continue
                    window = line[match.end() : match.end() + 180]
                    for value_match in VALUE_PATTERN.finditer(window):
                        raw_value = value_match.group("raw")
                        value, unit = parse_numeric_value(raw_value, metric_name)
                        if reject_candidate(metric_name, line, raw_value, value, unit):
                            continue
                        confidence = score_candidate(metric_name, alias, line, method, unit)
                        candidate = {
                            "metric_name": metric_name,
                            "metric_value": value,
                            "unit": unit,
                            "extraction_method": method,
                            "confidence": confidence,
                            "source_page": page.get("page_number"),
                            "evidence_snippet": compact_whitespace(line)[:320],
                        }
                        current = best_by_metric.get(metric_name)
                        if current is None or candidate["confidence"] > current["confidence"]:
                            best_by_metric[metric_name] = candidate
                        break

    full_text_l = compact_whitespace("\n".join(page.get("text", "") for page in pages)).lower()
    if "churn" not in best_by_metric and re.search(r"no\s+material\s+customer\s+churn|no\s+customer\s+churn", full_text_l):
        best_by_metric["churn"] = {
            "metric_name": "churn",
            "metric_value": 0.0,
            "unit": "%",
            "extraction_method": method,
            "confidence": 0.72 if method == "direct_parse" else 0.55,
            "source_page": None,
            "evidence_snippet": "No material customer churn was observed during the quarter.",
        }

    pipeline_candidate = extract_pipeline_from_text(pages, method)
    if pipeline_candidate:
        current = best_by_metric.get("pipeline")
        if current is None or pipeline_candidate["confidence"] > current["confidence"]:
            best_by_metric["pipeline"] = pipeline_candidate

    return [best_by_metric[name] for name in TARGET_METRICS if name in best_by_metric]


def infer_company_and_date(pdf_path: str | Path) -> tuple[str | None, str | None]:
    stem = Path(pdf_path).stem
    match = re.match(r"(?P<company>.+?)_(?P<quarter>Q[1-4])_(?P<year>20\d{2})$", stem, flags=re.IGNORECASE)
    if not match:
        return stem.replace("_", " "), None
    company = match.group("company").replace("_", " ")
    quarter = match.group("quarter").upper()
    year = int(match.group("year"))
    quarter_ends = {"Q1": "03-31", "Q2": "06-30", "Q3": "09-30", "Q4": "12-31"}
    return company, f"{year}-{quarter_ends[quarter]}"


def summarize_commentary(text: str, max_sentences: int = 3) -> str | None:
    normalized = compact_whitespace(text)
    if not normalized:
        return None
    heading_pattern = re.compile(
        r"(?:Operating Commentary|Performance Commentary|Business Commentary|Commentary|Pipeline & Business Update|Portfolio Composition Commentary)[:\s-]+(.+)",
        flags=re.IGNORECASE,
    )
    match = heading_pattern.search(normalized)
    candidate = match.group(1) if match else normalized
    sentences = re.split(r"(?<=[.!?])\s+", candidate)
    useful = [s.strip() for s in sentences if len(s.strip()) > 40]
    if not useful:
        return None
    return " ".join(useful[:max_sentences])[:700]

## 8. Direct PDF Extraction

Layer 1 reads the embedded text layer. `pdfplumber` is the first pass because it usually preserves page structure well enough for table-like reports. `pypdf` stays in as a fallback.

In [ ]:
def extract_pdf_text_pages(pdf_path: str | Path) -> tuple[list[dict[str, Any]], list[str]]:
    pdf_path = Path(pdf_path)
    errors: list[str] = []
    pages: list[dict[str, Any]] = []

    try:
        import pdfplumber
        with pdfplumber.open(pdf_path) as pdf:
            for idx, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                pages.append({"page_number": idx, "text": clean_text(text)})
    except Exception as exc:
        errors.append(f"pdfplumber failed: {exc}")

    if not any(page.get("text") for page in pages):
        try:
            from pypdf import PdfReader
            reader = PdfReader(str(pdf_path))
            pages = []
            for idx, page in enumerate(reader.pages, start=1):
                text = page.extract_text() or ""
                pages.append({"page_number": idx, "text": clean_text(text)})
        except Exception as exc:
            errors.append(f"pypdf failed: {exc}")

    return pages, errors


def parse_report_with_regex(pdf_path: str | Path, method: str = "direct_parse") -> dict[str, Any]:
    pages, errors = extract_pdf_text_pages(pdf_path)
    text = "\n".join(page.get("text", "") for page in pages)
    company_name, report_date = infer_company_and_date(pdf_path)
    metrics = extract_metrics_from_pages(pages, method=method) if text.strip() else []
    return {
        "company_name": company_name,
        "report_date": report_date,
        "raw_text": text,
        "pages": pages,
        "metrics": metrics,
        "commentary_summary": summarize_commentary(text),
        "errors": errors,
    }

example_pdf = sorted(PDF_INPUT_DIR.glob("*.pdf"))[0]
example_direct = parse_report_with_regex(example_pdf)
print(example_pdf.name)
print("Text chars:", len(example_direct["raw_text"]))
display(pd.DataFrame(example_direct["metrics"]))
print("Commentary:", example_direct["commentary_summary"])

## 9. OCR Fallback

Layer 2 converts pages to images and runs Tesseract OCR. This is only used when direct parsing returns little useful text or no metrics.

In [ ]:
def ocr_pdf_pages(pdf_path: str | Path, dpi: int = 200, max_pages: int = 3) -> tuple[list[dict[str, Any]], list[str]]:
    errors: list[str] = []
    try:
        from pdf2image import convert_from_path
        import pytesseract
        _ = pytesseract.get_tesseract_version()
    except Exception as exc:
        return [], [f"OCR unavailable: {exc}"]

    try:
        images = convert_from_path(str(pdf_path), dpi=dpi, first_page=1, last_page=max_pages)
    except Exception as exc:
        return [], [f"PDF to image conversion failed: {exc}"]

    pages = []
    for idx, image in enumerate(images, start=1):
        try:
            text = pytesseract.image_to_string(image) or ""
            pages.append({"page_number": idx, "text": clean_text(text)})
        except Exception as exc:
            errors.append(f"Tesseract failed on page {idx}: {exc}")
    return pages, errors


def parse_report_with_ocr(pdf_path: str | Path) -> dict[str, Any]:
    pages, errors = ocr_pdf_pages(pdf_path)
    text = "\n".join(page.get("text", "") for page in pages)
    company_name, report_date = infer_company_and_date(pdf_path)
    metrics = extract_metrics_from_pages(pages, method="ocr") if text.strip() else []
    return {
        "company_name": company_name,
        "report_date": report_date,
        "ocr_text": text,
        "ocr_pages": pages,
        "metrics": metrics,
        "commentary_summary": summarize_commentary(text),
        "errors": errors,
    }

ocr_pages, ocr_errors = ocr_pdf_pages(example_pdf, max_pages=1)
print("OCR pages returned:", len(ocr_pages))
print("OCR status:", ocr_errors[:1] if ocr_errors else "available")

## 10. Ollama GLM-OCR Fallback

`glm-ocr` runs locally through Ollama for the final visual fallback. The notebook first asks for text recognition, then retries with table recognition if the output is too thin. That keeps the page text readable for metric extraction instead of returning a partial table fragment.

In [ ]:
import base64
import html
import io
import json
import subprocess
import tempfile
import urllib.error
import urllib.request
from functools import lru_cache


OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "glm-ocr"
OLLAMA_KEEP_ALIVE = "10m"
ANSI_ESCAPE_RE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


@lru_cache(maxsize=1)
def ensure_ollama_model() -> None:
    """Pull glm-ocr once if it is not already available locally."""
    try:
        subprocess.run(
            ["ollama", "show", OLLAMA_MODEL],
            check=True,
            capture_output=True,
            text=True,
        )
        return
    except subprocess.CalledProcessError:
        pass
    except FileNotFoundError as exc:
        raise RuntimeError("Ollama CLI is not installed or not on PATH") from exc

    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)


def strip_ansi_escape_codes(text: str) -> str:
    return ANSI_ESCAPE_RE.sub("", text)


def html_table_to_text(text: str) -> str:
    if "<table" not in text.lower():
        return text
    text = html.unescape(text)
    text = re.sub(r"(?is)<(script|style).*?</\1>", " ", text)
    text = re.sub(r"(?i)</(td|th)>", " ", text)
    text = re.sub(r"(?i)<tr[^>]*>", "\n", text)
    text = re.sub(r"(?i)</tr>", "\n", text)
    text = re.sub(r"(?i)<br\s*/?>", "\n", text)
    text = re.sub(r"(?i)</?(table|thead|tbody|tfoot)>", "\n", text)
    text = re.sub(r"(?i)<[^>]+>", " ", text)
    lines = [compact_whitespace(line) for line in text.splitlines()]
    lines = [line for line in lines if line]
    return "\n".join(lines)


def normalize_glm_ocr_text(text: str) -> str:
    text = strip_ansi_escape_codes(text or "")
    text = text.replace("\r", "\n")
    text = text.replace("```markdown", "").replace("```", "")
    text = html_table_to_text(text)
    lines = []
    for raw_line in text.splitlines():
        line = compact_whitespace(raw_line)
        if not line:
            continue
        if line.lower() in {"markdown", "text", "table"}:
            continue
        lines.append(line)
    return "\n".join(lines)


def ollama_chat(payload_json: str) -> str:
    """Call the local Ollama chat API and return the model response text."""
    request = urllib.request.Request(
        f"{OLLAMA_BASE_URL}/api/chat",
        data=payload_json.encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=1800) as response:
        body = response.read().decode("utf-8")
    data = json.loads(body)
    return data.get("message", {}).get("content", "")


def ollama_run_cli(image_path: Path, prompt: str = "Text Recognition:") -> str:
    """Fallback to the interactive CLI form documented by Ollama."""
    result = subprocess.run(
        ["ollama", "run", OLLAMA_MODEL, prompt, str(image_path)],
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip() or result.stderr.strip()


def ollama_transcribe_prompt(image_rgb, prompt: str) -> str:
    with io.BytesIO() as buffer:
        image_rgb.save(buffer, format="PNG")
        image_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")

    payload = {
        "model": OLLAMA_MODEL,
        "messages": [
            {
                "role": "user",
                "content": prompt,
                "images": [image_b64],
            }
        ],
        "stream": False,
        "keep_alive": OLLAMA_KEEP_ALIVE,
        "options": {"num_predict": 768, "temperature": 0},
    }
    payload_json = json.dumps(payload)

    try:
        text = normalize_glm_ocr_text(ollama_chat(payload_json))
        if text:
            return text
    except (urllib.error.HTTPError, urllib.error.URLError):
        pass

    with tempfile.TemporaryDirectory() as tmpdir:
        page_png = Path(tmpdir) / "page.png"
        image_rgb.save(page_png)
        return normalize_glm_ocr_text(ollama_run_cli(page_png, prompt=prompt))


def transcribe_ollama_glm_ocr_page(image_rgb, prompt: str = "Text Recognition:") -> str:
    ensure_ollama_model()

    text_output = ollama_transcribe_prompt(image_rgb, prompt)
    if not text_output:
        return ""

    text_metrics = extract_metrics_from_pages([{"page_number": 1, "text": text_output}], method="glm_ocr")
    looks_thin = len(text_output) < 300 or len(text_metrics) == 0
    if not looks_thin:
        return text_output

    table_output = ollama_transcribe_prompt(image_rgb, "Table Recognition:")
    if not table_output:
        return text_output

    combined = normalize_glm_ocr_text("\n".join([text_output, table_output]))
    combined_metrics = extract_metrics_from_pages([{"page_number": 1, "text": combined}], method="glm_ocr") if combined else []
    if len(combined) > len(text_output) or combined_metrics:
        return combined
    return text_output


def ollama_glm_ocr_pdf_pages(pdf_path: str | Path, dpi: int = 200, max_pages: int = 3) -> tuple[list[dict[str, Any]], list[str]]:
    errors: list[str] = []
    try:
        from pdf2image import convert_from_path
    except Exception as exc:
        return [], [f"Ollama GLM-OCR unavailable: {exc}"]

    try:
        images = convert_from_path(str(pdf_path), dpi=dpi, first_page=1, last_page=max_pages)
    except Exception as exc:
        return [], [f"PDF to image conversion failed: {exc}"]

    pages: list[dict[str, Any]] = []
    for idx, image in enumerate(images, start=1):
        try:
            image_rgb = image.convert("RGB")
            transcription = transcribe_ollama_glm_ocr_page(image_rgb)
            pages.append({"page_number": idx, "text": clean_text(transcription)})
        except Exception as exc:
            errors.append(f"Ollama GLM-OCR failed on page {idx}: {exc}")

    return pages, errors


def parse_report_with_glm_ocr(pdf_path: str | Path) -> dict[str, Any]:
    pages, errors = ollama_glm_ocr_pdf_pages(pdf_path)
    text = "\n".join(page.get("text", "") for page in pages)
    company_name, report_date = infer_company_and_date(pdf_path)
    metrics = extract_metrics_from_pages(pages, method="glm_ocr") if text.strip() else []
    return {
        "company_name": company_name,
        "report_date": report_date,
        "glm_ocr_text": text,
        "glm_ocr_pages": pages,
        "metrics": metrics,
        "commentary_summary": summarize_commentary(text),
        "errors": errors,
    }


## 10.1 Ollama GLM-OCR Test

This is separate from the main pipeline. It runs a small sample of real PDF pages through `glm-ocr` directly so the notebook shows the VLM path working on its own.

In [ ]:
from pdf2image import convert_from_path

smoke_pdf_paths = sorted(PDF_INPUT_DIR.glob("*.pdf"))[:2]
smoke_rows: list[dict[str, Any]] = []
smoke_errors: list[dict[str, str]] = []

for pdf_path in smoke_pdf_paths:
    try:
        pages = convert_from_path(str(pdf_path), dpi=150, first_page=1, last_page=1)
        if not pages:
            smoke_errors.append({"source_file": pdf_path.name, "errors": "no page image returned"})
            continue
        transcription = transcribe_ollama_glm_ocr_page(pages[0].convert("RGB"))
        smoke_rows.append(
            {
                "source_file": pdf_path.name,
                "page": 1,
                "transcription_excerpt": transcription[:800],
                "char_count": len(transcription),
            }
        )
    except Exception as exc:
        smoke_errors.append({"source_file": pdf_path.name, "errors": str(exc)})

smoke_df = pd.DataFrame(smoke_rows)
print(f"Ran GLM-OCR smoke test on {len(smoke_rows)} page(s).")
if not smoke_df.empty:
    display(smoke_df)
if smoke_errors:
    print("Smoke-test notes:")
    display(pd.DataFrame(smoke_errors))


## 10.2 Text Overlay Comparison

This is a visual sanity check. The same page image is shown with the extracted text overlaid for direct parsing, Tesseract OCR, and Ollama GLM-OCR. It makes it easier to see whether the text layer, OCR, and vision fallback are all pointing at the same content.

In [ ]:
from pdf2image import convert_from_path
from textwrap import fill


def overlay_excerpt(text: str, max_lines: int = 12, max_chars: int = 1100) -> str:
    lines = [clean_text(line) for line in text.splitlines() if clean_text(line)]
    if not lines:
        return "[no extracted text]"

    interesting: list[str] = []
    alias_pool = [alias for aliases in METRIC_ALIASES.values() for alias in aliases]
    for line in lines:
        lower = line.lower()
        if any(alias in lower for alias in alias_pool) or re.search(r"\d", line):
            interesting.append(line)

    source_lines = interesting if interesting else lines
    excerpt = " ".join(source_lines[:max_lines])
    excerpt = compact_whitespace(excerpt)
    return excerpt[:max_chars] if excerpt else "[no extracted text]"


def glm_overlay_preview(pdf_path: Path, page_number: int = 1) -> dict[str, Any]:
    page_images = convert_from_path(str(pdf_path), dpi=160, first_page=page_number, last_page=page_number)
    if not page_images:
        return {
            "label": "Ollama GLM-OCR",
            "pages": [],
            "metrics": [],
            "errors": [f"No page image available for {pdf_path.name} page {page_number}."] ,
            "text": "",
        }

    transcription = ollama_transcribe_prompt(page_images[0].convert("RGB"), "Text Recognition:")
    pages = [{"page_number": page_number, "text": clean_text(transcription)}]
    return {
        "label": "Ollama GLM-OCR",
        "pages": pages,
        "metrics": extract_metrics_from_pages(pages, method="glm_ocr") if transcription.strip() else [],
        "errors": [],
        "text": transcription,
    }


def method_overlay_spec(pdf_path: Path) -> list[dict[str, Any]]:
    direct = parse_report_with_regex(pdf_path)
    ocr = parse_report_with_ocr(pdf_path)
    glm = glm_overlay_preview(pdf_path)

    return [
        {
            "label": "Direct parse",
            "pages": direct.get("pages", []),
            "metrics": direct.get("metrics", []),
            "errors": direct.get("errors", []),
            "text": direct.get("raw_text", ""),
        },
        {
            "label": "Tesseract OCR",
            "pages": ocr.get("ocr_pages", []),
            "metrics": ocr.get("metrics", []),
            "errors": ocr.get("errors", []),
            "text": ocr.get("ocr_text", ""),
        },
        {
            "label": "Ollama GLM-OCR",
            "pages": glm.get("glm_ocr_pages", []),
            "metrics": glm.get("metrics", []),
            "errors": glm.get("errors", []),
            "text": glm.get("glm_ocr_text", ""),
        },
    ]


def display_text_overlay_comparison(pdf_path: Path, page_number: int = 1) -> None:
    page_images = convert_from_path(str(pdf_path), dpi=160, first_page=page_number, last_page=page_number)
    if not page_images:
        print(f"No page image available for {pdf_path.name} page {page_number}.")
        return

    page_image = page_images[0].convert("RGB")
    specs = method_overlay_spec(pdf_path)

    fig, axes = plt.subplots(3, 1, figsize=(14, 24))
    summary_rows: list[dict[str, Any]] = []

    for ax, spec in zip(axes, specs):
        ax.imshow(page_image)
        ax.axis("off")
        if spec["label"] == "Ollama GLM-OCR":
            page_text = ollama_transcribe_prompt(page_image, "Text Recognition:")
        else:
            page_text = spec["text"] or (spec["pages"][0]["text"] if spec["pages"] else "")
        excerpt = overlay_excerpt(page_text)
        wrapped = fill(excerpt, width=74)
        metrics_found = len([m for m in spec["metrics"] if m.get("metric_name") in TARGET_METRICS])
        title = f"{spec['label']} | metrics: {metrics_found}"
        if spec["errors"]:
            title += " | notes present"
        ax.set_title(title, fontsize=11, pad=10)
        ax.text(
            0.03,
            0.97,
            wrapped,
            transform=ax.transAxes,
            va="top",
            ha="left",
            fontsize=8,
            color="white",
            bbox=dict(facecolor="black", alpha=0.68, boxstyle="round,pad=0.5"),
        )
        summary_rows.append(
            {
                "method": spec["label"],
                "text_chars": len(page_text or ""),
                "excerpt_chars": len(excerpt),
                "metrics_found": metrics_found,
                "notes": " | ".join(spec["errors"]) if spec["errors"] else "",
            }
        )

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.28)
    plt.show()
    display(pd.DataFrame(summary_rows))


comparison_pdf = sorted(PDF_INPUT_DIR.glob("*.pdf"))[0]
print(f"Overlay comparison PDF: {comparison_pdf.name}")
display_text_overlay_comparison(comparison_pdf)


## 11. LangChain Tool Wrappers and Optional ToolNode

The deterministic functions are wrapped as LangChain tools. In a more agentic version, a model could call these tools dynamically. In this PoC, the graph uses explicit nodes because extraction order and fallback policy should be predictable.

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode


@tool
def extract_text_tool(pdf_path: str) -> str:
    """Extract text from a PDF using direct parsing."""
    pages, _ = extract_pdf_text_pages(pdf_path)
    return "\n".join(page.get("text", "") for page in pages)


@tool
def ocr_tool(pdf_path: str) -> str:
    """Run OCR fallback on a PDF."""
    pages, errors = ocr_pdf_pages(pdf_path)
    if errors and not pages:
        return "\n".join(errors)
    return "\n".join(page.get("text", "") for page in pages)


tool_node = ToolNode([extract_text_tool, ocr_tool])
print("ToolNode configured with:", [t.name for t in [extract_text_tool, ocr_tool]])

## 12. LangGraph Orchestration

The graph stays deterministic:

1. Try direct parsing.
2. If that fails, try OCR.
3. If the result is still incomplete or suspicious, run GLM-OCR.
4. Validate against the current and prior period.

In [ ]:
from langgraph.graph import END, START, StateGraph


class ExtractionState(TypedDict, total=False):
    pdf_path: Required[str]
    company_name: str | None
    report_date: str | None
    raw_text: str
    ocr_text: str
    pages: list[dict[str, Any]]
    extracted: dict[str, Any]
    extraction_method: str
    confidence: float
    needs_review: bool
    review_reason: str | None
    errors: list[str]
    historical_metrics: dict[str, dict[str, Any]]
    vlm_attempted: bool
    enable_vlm: bool


def has_extracted_metrics(state: ExtractionState) -> bool:
    return bool(state.get("extracted", {}).get("metrics"))


def direct_parse_node(state: ExtractionState) -> ExtractionState:
    parsed = parse_report_with_regex(state["pdf_path"], method="direct_parse")
    metrics = parsed["metrics"]
    return {
        **state,
        "company_name": parsed["company_name"],
        "report_date": parsed["report_date"],
        "raw_text": parsed["raw_text"],
        "pages": parsed["pages"],
        "extracted": {
            "metrics": metrics,
            "commentary_summary": parsed["commentary_summary"],
        },
        "extraction_method": "direct_parse" if metrics else "direct_parse_no_metrics",
        "confidence": max([m["confidence"] for m in metrics], default=0.0),
        "errors": [*state.get("errors", []), *parsed.get("errors", [])],
    }


def ocr_node(state: ExtractionState) -> ExtractionState:
    parsed = parse_report_with_ocr(state["pdf_path"])
    existing = state.get("extracted", {})
    metrics = parsed["metrics"]
    commentary = parsed.get("commentary_summary") or existing.get("commentary_summary")
    return {
        **state,
        "ocr_text": parsed["ocr_text"],
        "extracted": {
            "metrics": metrics,
            "commentary_summary": commentary,
        },
        "extraction_method": "ocr" if metrics else "ocr_no_metrics",
        "confidence": max([m["confidence"] for m in metrics], default=state.get("confidence", 0.0)),
        "errors": [*state.get("errors", []), *parsed.get("errors", [])],
    }


def glm_ocr_node(state: ExtractionState) -> ExtractionState:
    parsed = parse_report_with_glm_ocr(state["pdf_path"])
    existing = state.get("extracted", {})
    existing_metrics = {m["metric_name"]: m for m in existing.get("metrics", [])}
    for metric in parsed["metrics"]:
        current = existing_metrics.get(metric["metric_name"])
        if current is None or current.get("flagged_for_review") or current.get("confidence", 0) < metric["confidence"]:
            existing_metrics[metric["metric_name"]] = metric
    commentary = parsed.get("commentary_summary") or existing.get("commentary_summary")
    return {
        **state,
        "extracted": {
            "metrics": [existing_metrics[name] for name in TARGET_METRICS if name in existing_metrics],
            "commentary_summary": commentary,
        },
        "extraction_method": "glm_ocr" if parsed["metrics"] else state.get("extraction_method", "glm_ocr_skipped"),
        "confidence": max([m["confidence"] for m in existing_metrics.values()], default=state.get("confidence", 0.0)),
        "needs_review": state.get("needs_review", False) or not bool(parsed["metrics"]),
        "review_reason": state.get("review_reason") or ("GLM-OCR returned no usable metrics" if not parsed["metrics"] else None),
        "errors": [*state.get("errors", []), *parsed.get("errors", [])],
        "vlm_attempted": True,
    }


def validate_node(state: ExtractionState) -> ExtractionState:
    metrics = state.get("extracted", {}).get("metrics", [])
    company_name = state.get("company_name")
    history = state.get("historical_metrics", {})
    by_metric = {m["metric_name"]: {**m} for m in metrics}
    needs_review = bool(state.get("needs_review", False))
    review_reasons: list[str] = []
    if not metrics:
        needs_review = True
        review_reasons.append("no metrics extracted from direct parse or OCR")

    for metric_name, metric in list(by_metric.items()):
        key = f"{company_name}|{metric_name}"
        prior = history.get(key)
        reason_parts = []
        value = metric.get("metric_value")
        if metric.get("extraction_method") in {"glm_ocr", "llm_vlm"}:
            reason_parts.append("vision fallback")
            metric["confidence"] = min(float(metric.get("confidence", 0.0)), 0.72)
        if not metric.get("evidence_snippet"):
            reason_parts.append("missing evidence snippet")
        if prior and value is not None and prior.get("metric_value") not in (None, 0):
            prior_value = float(prior["metric_value"])
            change = abs((float(value) - prior_value) / prior_value)
            if change > 0.50:
                reason_parts.append(f">50% change from prior value {prior_value:,.2f}")
        if reason_parts:
            metric["flagged_for_review"] = True
            metric["review_reason"] = "; ".join(reason_parts)
            needs_review = True
            review_reasons.append(f"{metric_name}: {metric['review_reason']}")
        else:
            metric["flagged_for_review"] = False
            metric["review_reason"] = None
        by_metric[metric_name] = metric

    for metric_name in TARGET_METRICS:
        key = f"{company_name}|{metric_name}"
        if metric_name not in by_metric and key in history:
            prior = history[key]
            by_metric[metric_name] = {
                "metric_name": metric_name,
                "metric_value": None,
                "unit": prior.get("unit"),
                "extraction_method": "missing_after_history",
                "confidence": 0.0,
                "source_page": None,
                "evidence_snippet": None,
                "flagged_for_review": True,
                "review_reason": "metric was previously reported but is missing in current extraction",
            }
            needs_review = True
            review_reasons.append(f"{metric_name}: missing after historical disclosure")

    return {
        **state,
        "extracted": {
            **state.get("extracted", {}),
            "metrics": [by_metric[name] for name in TARGET_METRICS if name in by_metric],
        },
        "needs_review": needs_review,
        "review_reason": "; ".join(review_reasons) if review_reasons else state.get("review_reason"),
    }


def route_after_direct_parse(state: ExtractionState) -> Literal["validate", "ocr"]:
    if has_extracted_metrics(state):
        return "validate"
    return "ocr"


def route_after_ocr(state: ExtractionState) -> Literal["validate"]:
    return "validate"


def route_after_validate(state: ExtractionState) -> Literal["glm_ocr", "end"]:
    if not state.get("enable_vlm") or state.get("vlm_attempted"):
        return "end"
    extracted_metrics = state.get("extracted", {}).get("metrics", [])
    if not extracted_metrics:
        return "glm_ocr"
    return "end"


builder = StateGraph(ExtractionState)
builder.add_node("direct_parse", direct_parse_node)
builder.add_node("ocr", ocr_node)
builder.add_node("glm_ocr", glm_ocr_node)
builder.add_node("validate", validate_node)

builder.add_edge(START, "direct_parse")
builder.add_conditional_edges("direct_parse", route_after_direct_parse, {"validate": "validate", "ocr": "ocr"})
builder.add_conditional_edges("ocr", route_after_ocr, {"validate": "validate"})
builder.add_edge("glm_ocr", "validate")
builder.add_conditional_edges("validate", route_after_validate, {"glm_ocr": "glm_ocr", "end": END})

graph = builder.compile()
print("LangGraph extraction graph compiled.")

## 13. Batch Folder Processing

Reports are processed in company/date order so the historical checks compare each metric to the prior value for the same company.

In [ ]:
def sort_key_for_pdf(path: Path) -> tuple[str, str, str]:
    company, report_date = infer_company_and_date(path)
    return (company or "", report_date or "9999-99-99", path.name)


def state_to_records(state: ExtractionState) -> list[dict[str, Any]]:
    source_file = Path(state["pdf_path"]).name
    records: list[dict[str, Any]] = []
    for metric in state.get("extracted", {}).get("metrics", []):
        record = MetricRecord(
            company_name=state.get("company_name"),
            report_date=state.get("report_date"),
            metric_name=metric.get("metric_name"),
            metric_value=metric.get("metric_value"),
            unit=metric.get("unit"),
            extraction_method=metric.get("extraction_method", state.get("extraction_method", "unknown")),
            confidence=float(metric.get("confidence", state.get("confidence", 0.0))),
            source_file=source_file,
            source_page=metric.get("source_page"),
            evidence_snippet=metric.get("evidence_snippet"),
            flagged_for_review=bool(metric.get("flagged_for_review", False)),
            review_reason=metric.get("review_reason"),
        )
        records.append(record.model_dump())

    commentary = state.get("extracted", {}).get("commentary_summary")
    if commentary:
        records.append(MetricRecord(
            company_name=state.get("company_name"),
            report_date=state.get("report_date"),
            metric_name="commentary_summary",
            metric_value=None,
            unit="text",
            extraction_method=state.get("extraction_method", "direct_parse"),
            confidence=0.74,
            source_file=source_file,
            source_page=None,
            evidence_snippet=commentary,
            flagged_for_review=False,
            review_reason=None,
        ).model_dump())
    return records


def process_folder(pdf_dir: Path, limit: int | None = None, enable_vlm: bool = True) -> tuple[pd.DataFrame, list[ExtractionState]]:
    pdf_paths = sorted(pdf_dir.glob("*.pdf"), key=sort_key_for_pdf)
    if limit:
        pdf_paths = pdf_paths[:limit]
    history: dict[str, dict[str, Any]] = {}
    states: list[ExtractionState] = []
    all_records: list[dict[str, Any]] = []

    for pdf_path in pdf_paths:
        initial_state: ExtractionState = {
            "pdf_path": str(pdf_path),
            "raw_text": "",
            "ocr_text": "",
            "extracted": {},
            "confidence": 0.0,
            "needs_review": False,
            "errors": [],
            "historical_metrics": history.copy(),
            "vlm_attempted": False,
            "enable_vlm": enable_vlm,
        }
        final_state = graph.invoke(initial_state)
        states.append(final_state)
        records = state_to_records(final_state)
        all_records.extend(records)

        for record in records:
            if record["metric_name"] in TARGET_METRICS and record["metric_value"] is not None:
                key = f"{record['company_name']}|{record['metric_name']}"
                history[key] = record

    df = pd.DataFrame(all_records)
    if not df.empty:
        df["metric_value"] = pd.to_numeric(df["metric_value"], errors="coerce")
        df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce").dt.date.astype("string")
    return df, states

print("Validation review rows above are from the actual PDF package.")

## 14. Process the Main PDF Folder

This section runs directly against the PDFs in the provided challenge package.

In [ ]:
main_df, main_states = process_folder(PDF_INPUT_DIR)
metrics_df = main_df.copy()
print(f"Processed {len(main_states)} PDFs from the configured challenge package.")
print("Ollama GLM-OCR is enabled by default, see the `enable_vlm` parameter in `process_folder`.")
print(f"Extracted {len(metrics_df)} long-table rows")

display(metrics_df.head(20)[[
    "company_name", "report_date", "metric_name", "metric_value", "unit", "extraction_method", "confidence", "source_file", "source_page", "flagged_for_review"
]])

error_rows = []
for state in main_states:
    if state.get("errors"):
        error_rows.append({"source_file": Path(state["pdf_path"]).name, "errors": " | ".join(state["errors"])})
if error_rows:
    print("Non-fatal extraction notes:")
    display(pd.DataFrame(error_rows).head(10))

## 15. SQLite Persistence

The SQLite layer simulates the first durable store for dashboarding. The table is long/narrow so adding a new metric does not require a schema migration.

In [ ]:
def persist_metrics_to_sqlite(df: pd.DataFrame, sqlite_path: Path) -> pd.DataFrame:
    sqlite_path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(sqlite_path)
    try:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS company_metrics (
                company_name TEXT,
                report_date TEXT,
                metric_name TEXT,
                metric_value REAL,
                unit TEXT,
                extraction_method TEXT,
                confidence REAL,
                source_file TEXT,
                source_page INTEGER,
                evidence_snippet TEXT,
                flagged_for_review INTEGER,
                review_reason TEXT,
                PRIMARY KEY (company_name, report_date, metric_name, source_file)
            )
            """
        )
        rows = df.where(pd.notnull(df), None).to_dict(orient="records")
        for row in rows:
            conn.execute(
                """
                INSERT OR REPLACE INTO company_metrics (
                    company_name, report_date, metric_name, metric_value, unit, extraction_method,
                    confidence, source_file, source_page, evidence_snippet, flagged_for_review, review_reason
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
                """,
                (
                    row.get("company_name"), row.get("report_date"), row.get("metric_name"), row.get("metric_value"),
                    row.get("unit"), row.get("extraction_method"), row.get("confidence"), row.get("source_file"),
                    row.get("source_page"), row.get("evidence_snippet"), int(bool(row.get("flagged_for_review"))),
                    row.get("review_reason"),
                ),
            )
        conn.commit()
        return pd.read_sql_query("SELECT * FROM company_metrics ORDER BY company_name, report_date, metric_name", conn)
    finally:
        conn.close()

stored_df = persist_metrics_to_sqlite(metrics_df, SQLITE_PATH)
print("SQLite database written to outputs/portfolio_metrics.sqlite")
print(f"Rows read back from SQLite: {len(stored_df)}")
display(stored_df.head(20))

## 16. Historical Validation and Anomaly Checks

The validation layer flags rows when:

- a metric was previously reported for the company but is missing now;
- a numeric value changes by more than 50% from the prior value;
- a row comes only from OCR or GLM-OCR and still looks weak;
- there is no evidence snippet.

In [ ]:
review_df = stored_df[stored_df["flagged_for_review"].astype(bool)].copy()
print(f"Flagged rows: {len(review_df)}")
if review_df.empty:
    print("No rows flagged by the validation rules in this run.")
else:
    display(review_df[[
        "company_name", "report_date", "metric_name", "metric_value", "unit", "extraction_method", "source_file", "review_reason", "evidence_snippet"
    ]])

print("The rows above are drawn from the actual PDF package.")

## 17. Pandas Review Table

This is the review table: one row per company-period-metric with method, confidence, source file/page, and evidence.

In [ ]:
review_columns = [
    "company_name", "report_date", "metric_name", "metric_value", "unit", "extraction_method",
    "confidence", "source_file", "source_page", "evidence_snippet", "flagged_for_review", "review_reason",
]
review_table = stored_df[review_columns].sort_values(["company_name", "report_date", "metric_name"])
display(review_table.head(50))

numeric_metrics = stored_df[stored_df["metric_name"].isin(TARGET_METRICS)].copy()
pivot = numeric_metrics.pivot_table(
    index=["company_name", "report_date", "source_file"],
    columns="metric_name",
    values="metric_value",
    aggfunc="first",
).reset_index()
display(pivot.head(20))

## 18. Basic Visualization

The charts are intentionally simple. The same table can feed a dashboard later.

In [ ]:
def plot_metric_timeseries(df: pd.DataFrame, metric_name: str, title: str) -> None:
    plot_df = df[(df["metric_name"] == metric_name) & df["metric_value"].notna()].copy()
    if plot_df.empty:
        print(f"No data to plot for {metric_name}.")
        return
    plot_df["report_date_dt"] = pd.to_datetime(plot_df["report_date"], errors="coerce")
    plot_df = plot_df.dropna(subset=["report_date_dt"])
    if plot_df.empty:
        print(f"No dated data to plot for {metric_name}.")
        return
    plt.figure(figsize=(10, 5))
    for company, group in plot_df.groupby("company_name"):
        group = group.sort_values("report_date_dt")
        plt.plot(group["report_date_dt"], group["metric_value"] / 1_000_000, marker="o", label=company)
    plt.title(title)
    plt.ylabel("Value ($M)")
    plt.xlabel("Report date")
    plt.legend(loc="best", fontsize=8)
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

plot_metric_timeseries(stored_df, "revenue", "Revenue over time by company")
plot_metric_timeseries(stored_df, "arr", "ARR over time by company")

flag_counts = stored_df.groupby("company_name")["flagged_for_review"].sum().sort_values(ascending=False)
plt.figure(figsize=(8, 4))
flag_counts.plot(kind="bar")
plt.title("Flagged metrics by company")
plt.ylabel("Flagged rows")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 19. Simple Forecasting Baseline

This forecast reads the saved metrics back from SQLite and fits a regularized linear trend per company and metric. I am using Ridge regression because the series in this PoC are short, so a heavier time-series model would overfit quickly. 

The model predicts the next reporting point from the observed order of period. The result is directional only and should be treated as a weak baseline until there is more history per company.


In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def read_metrics_from_sqlite(sqlite_path: Path) -> pd.DataFrame:
    conn = sqlite3.connect(sqlite_path)
    try:
        return pd.read_sql_query("SELECT * FROM company_metrics", conn)
    finally:
        conn.close()


def infer_display_scale(metric_name: str, unit: str | None) -> tuple[float, str]:
    unit_text = (unit or "").strip().lower()
    if unit_text in {"%", "percent", "percentage"} or metric_name in {"gross_margin", "churn"}:
        return 1.0, "%"
    if metric_name == "headcount" or unit_text in {"count", "people", "employees", "fte"}:
        return 1.0, "count"
    return 1_000_000.0, "$M"


def build_company_metric_forecasts(sqlite_path: Path, metric_names: list[str] | None = None, min_points: int = 2) -> pd.DataFrame:
    history = read_metrics_from_sqlite(sqlite_path)
    history = history[history["metric_value"].notna()].copy()
    history["report_date_dt"] = pd.to_datetime(history["report_date"], errors="coerce")
    history = history.dropna(subset=["report_date_dt"])
    if metric_names is not None:
        history = history[history["metric_name"].isin(metric_names)].copy()

    forecast_rows: list[dict[str, object]] = []
    for (company, metric), group in history.groupby(["company_name", "metric_name"]):
        ordered = group.sort_values(by=["report_date_dt", "source_file"]).copy()
        series = (
            ordered.groupby("report_date_dt", as_index=False)
            .agg(metric_value=("metric_value", "last"))
            .sort_values(by="report_date_dt")
            .reset_index(drop=True)
        )
        if series.empty:
            continue

        unit_series = group["unit"].dropna().astype(str)
        unit = unit_series.iloc[0] if not unit_series.empty else None
        scale, display_unit = infer_display_scale(metric, unit)
        last_actual_value = float(series["metric_value"].iloc[-1])
        last_actual_date = pd.to_datetime(series["report_date_dt"].iloc[-1])

        if len(series) >= min_points:
            X = np.arange(len(series)).reshape(-1, 1)
            y = series["metric_value"].astype(float).to_numpy()
            model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
            model.fit(X, y)
            forecast_value = float(model.predict(np.array([[len(series)]], dtype=float))[0])
            model_name = "ridge_regression"
            model_note = f"Regularized linear trend on {len(series)} historical points."
        else:
            forecast_value = last_actual_value
            model_name = "naive_last_observation"
            model_note = f"Only {len(series)} historical point(s); using last observation as a placeholder."

        if len(series) >= 2:
            deltas = series["report_date_dt"].diff().dropna()
            step = deltas.median() if not deltas.empty else pd.Timedelta(days=90)
        else:
            step = pd.Timedelta(days=90)

        forecast_date = last_actual_date + step
        forecast_rows.append(
            {
                "company_name": company,
                "metric_name": metric,
                "history_points": int(len(series)),
                "last_actual_date": last_actual_date.date().isoformat(),
                "last_actual_value": last_actual_value,
                "forecast_date": forecast_date.date().isoformat(),
                "forecast_value": forecast_value,
                "unit": unit or display_unit,
                "display_unit": display_unit,
                "model": model_name,
                "model_note": model_note,
                "scaled_last_actual": last_actual_value / scale,
                "scaled_forecast_value": forecast_value / scale,
            }
        )

    forecast_df = pd.DataFrame(forecast_rows)
    return forecast_df.sort_values(["company_name", "metric_name"]).reset_index(drop=True)


forecast_df = build_company_metric_forecasts(SQLITE_PATH, metric_names=TARGET_METRICS, min_points=2)
print(f"Forecast rows: {len(forecast_df)}")
if forecast_df.empty:
    print("No forecasts were produced because the SQLite table does not yet have enough history.")
else:
    display(
        forecast_df[
            [
                "company_name",
                "metric_name",
                "history_points",
                "last_actual_date",
                "last_actual_value",
                "forecast_date",
                "forecast_value",
                "unit",
                "model",
                "model_note",
            ]
        ]
    )


def plot_history_and_forecast(sqlite_path: Path, forecast_df: pd.DataFrame, metric_name: str) -> None:
    history = read_metrics_from_sqlite(sqlite_path)
    history = history[(history["metric_name"] == metric_name) & history["metric_value"].notna()].copy()
    if history.empty:
        print(f"No history available for {metric_name}.")
        return
    history["report_date_dt"] = pd.to_datetime(history["report_date"], errors="coerce")
    history = history.dropna(subset=["report_date_dt"])
    if history.empty:
        print(f"No dated history available for {metric_name}.")
        return

    forecast_subset = forecast_df[forecast_df["metric_name"] == metric_name].copy()
    if forecast_subset.empty:
        print(f"No forecast available for {metric_name}.")
        return

    scale, display_unit = infer_display_scale(
        metric_name,
        history["unit"].dropna().iloc[0] if not history["unit"].dropna().empty else None,
    )
    fig, ax = plt.subplots(figsize=(11, 5))
    forecast_label_added = False
    for company, group in history.groupby("company_name"):
        group = group.sort_values("report_date_dt")
        line, = ax.plot(
            group["report_date_dt"],
            group["metric_value"] / scale,
            marker="o",
            linewidth=1.8,
            label=f"{company} actual",
        )
        forecast_row = forecast_subset[forecast_subset["company_name"] == company]
        if forecast_row.empty:
            continue
        forecast_row = forecast_row.iloc[0]
        forecast_date = pd.to_datetime(forecast_row["forecast_date"])
        ax.plot(
            [group["report_date_dt"].iloc[-1], forecast_date],
            [group["metric_value"].iloc[-1] / scale, forecast_row["forecast_value"] / scale],
            linestyle="--",
            color=line.get_color(),
            alpha=0.75,
        )
        forecast_label = "Forecast" if not forecast_label_added else None
        ax.scatter(
            [forecast_date],
            [forecast_row["forecast_value"] / scale],
            marker="*",
            s=180,
            color=line.get_color(),
            edgecolor="black",
            linewidth=0.4,
            zorder=3,
            label=forecast_label,
        )
        forecast_label_added = True

    ax.set_title(f"{metric_name.title()} history and next-period forecast")
    ax.set_xlabel("Report date")
    ax.set_ylabel(display_unit or "Value")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()


for metric in ["revenue", "arr"]:
    plot_history_and_forecast(SQLITE_PATH, forecast_df, metric)


## 20. Discussion of Tradeoffs

**Why start with direct parsing**

The provided PDFs already have a usable text layer, so direct parsing is the cheapest and easiest path to audit.

**Why keep OCR and GLM-OCR**

Some reporting packages will be scanned, image-heavy, or formatted in a way that breaks text extraction. OCR covers the common scanned-PDF case. GLM-OCR is the last pass when the text layer and OCR are not enough.

**Why use LangGraph**

The fallback order should be obvious. LangGraph makes the state and routing explicit instead of hiding it inside a general-purpose agent.

**Limits**

- Portfolio summary PDFs are treated as single documents in this version.
- Label-based extraction will miss some layouts.
- OCR quality depends on local Tesseract and Poppler.
- The validation rules are basic by design.

## 21. Next Steps

To turn this into an internal dashboard, I would keep the same extraction table and add a thin reporting layer on top of it:

- persist the long table in Postgres instead of SQLite and keep source page, snippet, and extraction method as first-class fields;
- add an analyst review queue for low-confidence or missing-after-history rows;
- expose a small API that returns company, period, metric, confidence, and evidence for a dashboard client;
- build views for trend lines, quarter-over-quarter change, and outlier flags by company;
- add filters for company, strategy, period, metric, and review status;
- add scheduled ingestion so new PDFs flow in automatically and reruns are tracked;
- add LangSmith tracing and a labeled test set so extraction quality can be monitored over time.
- add a forecasting layer that learns from more quarterly history; the current Ridge baseline is weak because the series in this PoC are short.